## Import and format the data to build a DataLoader

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam # we will use Adam instead of Stochastic Gradient Decent. 
import lightning as L 
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path


In [2]:
url =  Path.cwd()/ "iris.txt"
df = pd.read_table(url, sep=",", header=None)

df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df.columns = ["sepal_length",
              "sepal_width",
              "petal_length",
              "petal_width",
              "class"]

In [4]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [5]:
df.shape

(150, 5)

In [6]:
df["class"].unique()

<StringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str

In [7]:
# We see if the dataset is balanced
for class_name in df["class"].unique():
    print(class_name, ":", sum(df["class"] == class_name), sep="")

Iris-setosa:50
Iris-versicolor:50
Iris-virginica:50


In [8]:
# For simplicity we will use only 'petal_width' and 'sepal_width' as inputs
input_values = df[['petal_width', 'sepal_width']]
input_values.head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [9]:
label_values = df['class']
label_values.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: class, dtype: str

In [10]:
# Because the Neural Network expects values to be numbers we convert 'label_values' into numbers using factorize()
class_as_numbers = label_values.factorize()[0] # factorize() returns a list of lists. We need the first one so we use [0]
class_as_numbers

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [11]:
input_train, input_test, label_train, label_test = train_test_split(input_values,
                                                                    class_as_numbers,
                                                                    test_size=0.25,
                                                                    stratify=class_as_numbers,
                                                                    random_state=42)

In [12]:
input_train.shape

(112, 2)

In [13]:
label_train.shape

(112,)

In [14]:
input_test.shape

(38, 2)

In [15]:
label_test.shape

(38,)

Because the Neural Network will have 3 outputs we need to convert the numbers in label_train into 3 element arrays where each element in the array corresponds to a specific output in the neural network. 
We will use [1.0, 0.0, 0.0] for Setosa, [0.0, 1.0, 0.0] for Versicolor, and [0.0, 0.0, 1.0] for Virginica


In [16]:
# we will use one-hot encoding for that. We create a new tensor with oen-hot encoded rows for each row in the original dataset.
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

In [17]:
one_hot_label_train[:10]

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]])

In [18]:
# We will normalize the input variables in the range 0 to 1.
# First we determine the maximum and minimum values in input_train.
max_vals_in_input_train = input_train.max()
max_vals_in_input_train

petal_width    2.5
sepal_width    4.4
dtype: float64

In [19]:
min_vals_in_input_train = input_train.min()
min_vals_in_input_train

petal_width    0.1
sepal_width    2.0
dtype: float64

In [20]:
# Next we normalize input_train with max and min values from input_train
input_train = (input_train - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_train.head()

,petal_width,sepal_width
130,0.750000,0.333333
122,0.791667,0.333333
81,0.375000,0.166667
71,0.500000,0.333333
89,0.500000,0.208333


In [21]:
# And then we normalize input_test with max and min values from input_train
input_test = (input_test - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
input_test.head()

,petal_width,sepal_width
42,0.041667,0.500000
56,0.625000,0.541667
99,0.500000,0.333333
53,0.500000,0.125000
38,0.041667,0.416667


In [22]:
# We load the training data into a 'DataLoader', which we will use to train the neural network.
#  To do that we first convert 'input_train' into tensors. Then we combine 'input_train' with 'one_hot_label_train' to create 'TensorDataset'.
# Finally we use 'TensorDataset' to create the 'DataLoader'.

We load the training data into a 'DataLoader', which we will use to train the neural network.
To do that we first convert 'input_train' into tensors. Then we combine 'input_train' with 'one_hot_label_train' to create 'TensorDataset'.
Finally we use 'TensorDataset' to create the 'DataLoader'.

In [23]:
# We will use .values to pass the values instead of the dataframe.
# We will also use .type(torch.float32) to make sure the numbers are saved in the correct format for the neural network to process them.

input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_train_tensors[:5]

tensor([[0.7500, 0.3333],
        [0.7917, 0.3333],
        [0.3750, 0.1667],
        [0.5000, 0.3333],
        [0.5000, 0.2083]])

In [24]:
# we also convert to tensors the 'input_test'
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)
input_test_tensors[:5] 

tensor([[0.0417, 0.5000],
        [0.6250, 0.5417],
        [0.5000, 0.3333],
        [0.5000, 0.1250],
        [0.0417, 0.4167]])

In [25]:
# we have already the one-hot encoded 'class' in 'label_train'

train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

## Building the Neural Network

In [26]:
class myNN(L.LightningModule):
    def __init__(self):
        super().__init__()
        L.seed_everything(42)
        
        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
        self.loss = nn.MSELoss(reduction='sum')

    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))
        return output_values
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.001)
    
    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)
        return loss
    


In [27]:
model = myNN()
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

Seed set to 42


input_to_hidden.weight tensor([[ 0.5400,  0.5900],
        [-0.1700,  0.6500]])
input_to_hidden.bias tensor([-0.1500,  0.1400])
hidden_to_output.weight tensor([[-0.3400,  0.4200],
        [ 0.6200, -0.5200],
        [ 0.6100,  0.1300]])
hidden_to_output.bias tensor([0.5200, 0.1000, 0.3400])


## Training the Neural Network

In [28]:
model = myNN()

Seed set to 42


In [29]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enabl

Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 457.95it/s, v_num=5]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 112/112 [00:00<00:00, 449.57it/s, v_num=5]


In [30]:
predictions = model(input_test_tensors)

In [31]:
predictions[0:4,]

tensor([[0.7585, 0.0282, 0.0331],
        [0.1994, 0.3877, 0.4948],
        [0.2486, 0.3490, 0.3868],
        [0.1941, 0.3776, 0.3712]], grad_fn=<SliceBackward0>)

In [32]:
# We select the output with the highest value from each array.
predicted_labels = torch.argmax(predictions, dim=1) # dim=0 applies argmax to rows, dim=1 applies argmax to columns
predicted_labels[0:3] # we print the first 3 predictions

tensor([0, 2, 2])

In [33]:
# The first is 0 which is setosa, the second and third are 2 which is virginica.

# We calculate the 'predicted_labels' to the known values ('label_test')

torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.7368)

In [34]:
# The Neural Network predicted correctly 73% of the time.

The Neural Network predicted correctly 73% of the time. We will train for longer to try to achieve better result. 'Lightning' creates checkpoint files with weights and biases of the training we have done so far so we dont have to train from the beginning. We will do another 90 epochs to reach 100 in total

In [35]:
path_to_checkpoint = trainer.checkpoint_callback.best_model_path #by default 'best' == 'most recent'

In [36]:
trainer = L.Trainer(max_epochs=100)
trainer.fit(model, train_dataloaders=train_dataloader, ckpt_path=path_to_checkpoint)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/odysseasgeorgiades/Desktop/Neural Networks/03-Multi-input-output-nn/lightning_logs/version_5/checkpoints/epoch=9-step=1120.ckpt
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/odysseasgeorgiades/Desktop/Neural Networks/03-Multi-input-output-nn/lightning_logs/version_5/checkpoints' to '/Users/odysseasgeorgi

Epoch 99: 100%|██████████| 112/112 [00:00<00:00, 480.13it/s, v_num=6]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 112/112 [00:00<00:00, 470.91it/s, v_num=6]


In [37]:
# We calculate the accuracy again.
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.9211)

This time after 100 epochs we classified correctly 92% of the test set.

## Making Predictions with New Data

In [38]:
input_sample = pd.Series([0.2, 3.0], index=min_vals_in_input_train.index)
normalized_values = (input_sample - min_vals_in_input_train) / (max_vals_in_input_train - min_vals_in_input_train)
tensor_input = torch.tensor(normalized_values.values, dtype=torch.float32).unsqueeze(0)

with torch.no_grad():
    output = model(tensor_input)
    predicted_class = torch.argmax(output, dim=1)
    print(predicted_class.tolist())



[0]


We predicted 0. That means it is Setosa